In [1]:
import os
from datetime import UTC, datetime

import biogeme.biogeme as bio
import biogeme.biogeme_logging as blog
import biogeme.database as db
import numpy as np
import pandas as pd
from biogeme import models
from biogeme.expressions import Beta, Variable, log

/home/yianzhang/work/research/migration/migration/.pixi/envs/default/lib/python3.12/site-packages/tqdm_joblib/__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
unixtime = int(datetime.now(UTC).timestamp())

In [3]:
year = 2018

In [4]:
df = pd.read_parquet(f"../data/us_estdata_{year}.parquet")
df

,RT,SERIALNO,DIVISION,SPORDER,PUMA,REGION,ST,ADJINC,AGEP,CIT,...,ALT200_NAICS_GROUP_PCT_AGR_EXT,ALT200_NAICS_GROUP_PCT_HIGH_ED,ALT200_NAICS_GROUP_PCT_LICENSE,ALT200_NAICS_GROUP_PCT_GOODS_TRADE,ALT200_NAICS_GROUP_PCT_LOW_SKILL_SVC,ALT200_NAICS_GROUP_PCT_GOVT,ALT200_OWN_AGE_PCT,ALT200_OWN_RACE_PCT,ALT200_PUMA,ALT200_STATE
0,P,2018HU0637969,3,1,1104,2,17,1013097,61,1,...,0.003015,0.444965,0.049476,0.132399,0.353274,0.016870,0.451164,0.639904,1301005,13
1,P,2018HU0808038,5,5,505,3,24,1013097,70,4,...,0.000116,0.496820,0.055431,0.109353,0.337002,0.001278,0.149475,0.030880,2401203,24
2,P,2018HU1051188,9,1,7310,4,6,1013097,37,1,...,0.012585,0.550808,0.034583,0.166458,0.188652,0.046914,0.335738,0.847018,2500200,25
3,P,2018HU0461416,2,1,4110,1,36,1013097,55,4,...,0.006161,0.377409,0.037022,0.297105,0.256795,0.025508,0.369298,0.889449,2101500,21
4,P,2018GQ0016994,5,1,51167,3,51,1013097,40,1,...,0.006331,0.361470,0.015562,0.265037,0.290362,0.061238,0.368173,0.242046,3705100,37
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253085,P,2018HU0435287,5,2,1400,3,24,1013097,45,1,...,0.025628,0.276017,0.048273,0.285323,0.313259,0.051500,0.394388,0.936756,2600400,26
253086,P,2018HU0652247,8,2,900,4,8,1013097,23,1,...,0.004092,0.280297,0.040778,0.435102,0.220117,0.019613,0.211184,0.901713,3902400,39
253087,P,2018HU0192642,8,1,824,4,8,1013097,57,1,...,0.000357,0.473350,0.036659,0.199976,0.201097,0.088562,0.366196,0.341387,2101701,21
253088,P,2018HU0750302,7,1,5915,3,48,1013097,54,1,...,0.042082,0.171070,0.031904,0.478843,0.250564,0.025537,0.396090,0.885657,2600900,26


In [5]:
cat_cols = df.dtypes[df.dtypes == "category"].keys()
df[cat_cols] = df[cat_cols].apply(lambda x: x.astype(int))
# convert all relevant strings to integers
obj_cols = list(df.dtypes[df.dtypes == "object"].keys())
obj_cols.remove("RT")
obj_cols.remove("SERIALNO")
obj_cols.remove("TYPE.ORIG")
df[obj_cols] = df[obj_cols].apply(lambda x: x.astype(int))

In [6]:
# need to have the sentinel values be different so that SAME_CBSA works correctly
df["NAME_NUM.ORIG"].min(), df["ALT1_CBSA"].min()

(np.int64(-2), np.int16(-1))

In [7]:
# clean up the database (Biogeme Database can only have numerical values)
df_train = df.select_dtypes(["number"])
assert df_train.isna().sum().sum() == 0
del df

In [8]:
# defining the chosen alterantive for each person explicitly
# each person has chosen from alterantives 0-200, 0 represents staying and 1-200 represent moving to the PUMAs each represents
# for movers, they all choose alterantive 1 by construction; all stayers choose alterantive 0
df_train["ALT_CHOICE"] = 0
for i in range(1, 201):
    var = "ALT" + str(i) + "_PUMA"
    df_train["ALT_CHOICE"] = np.where(
        df_train[var] == df_train["CHOSEN"], i, df_train["ALT_CHOICE"]
    )
df_train["ALT_CHOICE"] = np.where(df_train["STAY"] == 1, 0, df_train["ALT_CHOICE"])

In [9]:
df_train["ALT_CHOICE"].value_counts()

ALT_CHOICE
0    220046
1     33044
Name: count, dtype: int64

In [10]:
# df.loc[df["CBSA_NAME_ORIG"] == -1, "CBSA_NAME_ORIG"] = -2

In [11]:
# df["INTERNAL"] = df["ORIGIN"] == df["CHOSEN"]

In [12]:
# df["MIGSP_ORIG"] = df["ORIGIN"].astype(str).str.zfill(7).str[0:2].astype(int)

In [ ]:
# making the Biogeme Database that is used for the model estimation
database = db.Database("us_data", df_train)

In [ ]:
num_alternatives = 200

In [ ]:
c_stay = Beta("c_stay", 0, None, None, 0)

# age
c_stay_age_18_22 = Beta("c_stay_age_18_22", 0, None, None, 0)
c_stay_age_23_29 = Beta("c_stay_age_23_29", 0, None, None, 0)
c_stay_age_30_39 = Beta("c_stay_age_30_39", 0, None, None, 0)
c_stay_age_40_49 = Beta("c_stay_age_40_49", 0, None, None, 0)
c_stay_age_50_64 = Beta("c_stay_age_50_64", 0, None, None, 0)
# c_stay_age_65 = Beta("c_stay_age_65", 0, None, None, 1)

c_stay_proportion_18_34 = Beta("c_stay_proportion_18_22", 0, None, None, 0)
c_stay_proportion_35_64 = Beta("c_stay_proportion_35_64", 0, None, None, 0)
c_stay_proportion_65_plus = Beta("c_stay_proportion_65_plus", 0, None, None, 0)

# personal lives
c_stay_child_under_6 = Beta("c_stay_child_under_6", 0, None, None, 0)
c_stay_child_6_to_17 = Beta("c_stay_child_6_to_17", 0, None, None, 0)
c_stay_proportion_hh_with_children = Beta(
    "c_stay_proportion_hh_with_children", 0, None, None, 0
)
c_stay_married_more_than_year = Beta("c_stay_married_more_than_year", 0, None, None, 0)
c_stay_married_less_than_year = Beta("c_stay_married_less_than_year", 0, None, None, 0)
c_stay_recently_divorced_or_widowed = Beta(
    "c_stay_recently_divorced_or_widowed", 0, None, None, 0
)
c_stay_2work_mar = Beta("c_stay_2work_mar", 0, None, None, 0)
c_stay_1work_mar = Beta("c_stay_1work_mar", 0, None, None, 0)
c_stay_single_parent = Beta("c_stay_single_parent", 0, None, None, 0)

# local characteristics
c_stay_T34 = Beta("c_stay_t34", 0, None, None, 0)
c_stay_metro = Beta("c_stay_metro", 0, None, None, 0)
c_stay_micro = Beta("c_stay_micro", 0, None, None, 0)
# c_stay_density = Beta("c_stay_density", 0.00000437, None, None, 0)

# education
c_stay_edu_college = Beta("c_stay_edu_college", 0, None, None, 0)
c_stay_edu_high = Beta("c_stay_edu_high", 0, None, None, 0)
c_stay_edu_nohigh = Beta("c_stay_edu_nohigh", 0, None, None, 0)

c_stay_in_college = Beta("c_stay_in_college", 0, None, None, 0)
c_stay_proportion_in_college = Beta("c_stay_proportion_in_college", 0, None, None, 0)

# race & foreign status
c_stay_foreign = Beta("c_stay_foreign", 0, None, None, 0)

c_stay_proportion_same_race_black = Beta(
    "c_stay_proportion_same_race_black", 0, None, None, 0
)
c_stay_proportion_same_race_aapi = Beta(
    "c_stay_proportion_same_race_aapi", 0, None, None, 0
)
c_stay_proportion_same_race_indian = Beta(
    "c_stay_proportion_same_race_indian", 0, None, None, 0
)
c_stay_proportion_same_race_latino = Beta(
    "c_stay_proportion_same_race_latino", 0, None, None, 0
)
c_stay_proportion_foreign = Beta("c_stay_proportion_foreign", 0, None, None, 0)

# economic & housing indicators
c_stay_hh_income = Beta("c_stay_hh_income", 0, None, None, 0)
c_stay_proportion_struggling = Beta("c_stay_proportion_strugglin", 0, None, None, 0)
c_stay_yrs_since_median_structure_built = Beta(
    "c_stay_yrs_since_median_structure_built", 0, None, None, 0
)
c_stay_median_house_value_over_median_income = Beta(
    "c_stay_median_house_value_over_median_income", 0, None, None, 0
)
c_stay_median_gross_rent_percentage_hh_inc = Beta(
    "c_stay_median_gross_rent_percentage_hh_inc", 0, None, None, 0
)
c_stay_median_owner_costs = Beta("c_stay_median_owner_costs", 0, None, None, 0)
c_stay_unemp_rate = Beta("c_stay_unemp", 0, None, None, 0)
c_stay_vacancy_rate = Beta("c_stay_vacancy_rate", 0, None, None, 0)

# work indicators
c_stay_median_travel_time = Beta("c_stay_median_travel_time", 0, None, None, 0)
c_stay_proportion_alt_commute = Beta("c_stay_proportion_alt_commute", 0, None, None, 0)

# NOTE: this assumes that NAICS code stays constant between the origin and destination
c_stay_mil = Beta("c_stay_mil", 0, None, None, 0)
c_stay_naics_govt = Beta("c_stay_naics_govt", 0, None, None, 0)
c_stay_naics_goods_trade = Beta("c_stay_naics_goods_trade", 0, None, None, 0)
c_stay_naics_license = Beta("c_stay_naics_license", 0, None, None, 0)
c_stay_naics_high_ed = Beta("c_stay_naics_high_ed", 0, None, None, 0)
c_stay_naics_agr_ext = Beta("c_stay_naics_agr_ext", 0, None, None, 0)
# low svc proportion skipped; shouldn't be very relevant

c_stay_proportion_mil = Beta("c_stay_proportion_mil", 0, None, None, 0)
c_stay_naics_proportion_govt = Beta("c_stay_naics_proportion_govt", 0, None, None, 0)
c_stay_naics_proportion_goods_trade = Beta(
    "c_stay_naics_proportion_goods_trade", 0, None, None, 0
)
c_stay_naics_proportion_license = Beta(
    "c_stay_proportion_naics_license", 0, None, None, 0
)
c_stay_naics_proportion_high_ed = Beta(
    "c_stay_proportion_naics_high_ed", 0, None, None, 0
)
c_stay_naics_proportion_agr_ext = Beta(
    "c_stay_proportion_naics_agr_ext", 0, None, None, 0
)


In [ ]:
# dictionary of utilities; number -> utility function
V = {}

In [ ]:
# defining the staying utility function
V[0] = (
    c_stay
    # age
    + c_stay_age_18_22 * Variable("AGE_18_22")
    + c_stay_age_23_29 * Variable("AGE_23_29")
    + c_stay_age_30_39 * Variable("AGE_30_39")
    + c_stay_age_40_49 * Variable("AGE_40_49")
    + c_stay_age_50_64 * Variable("AGE_50_64")
    # + c_stay_age_65 * Variable("AGE_OVER_65")
    + c_stay_proportion_18_34 * Variable("Proportion of people 18-34.ORIG")
    + c_stay_proportion_35_64 * Variable("Proportion of people 35-64.ORIG")
    + c_stay_proportion_65_plus * Variable("Proportion of people 65+.ORIG")
    # personal lives
    + c_stay_child_under_6 * Variable("CHILD_UNDER_6")
    + c_stay_child_6_to_17 * Variable("CHILD_6_TO_17")
    + c_stay_proportion_hh_with_children
    * Variable("Proportion of households with children.ORIG")
    * Variable("CHILD")
    + c_stay_married_more_than_year * Variable("MARRIED_MORE_THAN_YEAR")
    + c_stay_married_less_than_year * Variable("RECENTLY_MARRIED")
    + c_stay_recently_divorced_or_widowed * Variable("RECENTLY_WIDOWED_OR_DIVORCED")
    + c_stay_2work_mar * Variable("WORK2_MAR")
    + c_stay_1work_mar * Variable("WORK1_MAR")
    + c_stay_single_parent * Variable("SINGLE_PARENT")
    # local characteristics
    + c_stay_T34 * (Variable("TYPE_NUM.ORIG") == 0)
    + c_stay_metro * (Variable("TYPE_NUM.ORIG") == 1)
    + c_stay_micro * (Variable("TYPE_NUM.ORIG") == 2)
    # education
    + c_stay_edu_college * Variable("EDU_BACHELORS")
    + c_stay_edu_high * Variable("EDU_HIGH")
    + c_stay_edu_nohigh * Variable("EDU_NOHIGH")
    + c_stay_in_college * Variable("IN_COLLEGE")
    + c_stay_proportion_in_college
    * Variable("Proportion of people in college.ORIG")
    * Variable("IN_COLLEGE")
    # race & foreign status
    + c_stay_foreign * Variable("FOREIGN")
    + c_stay_proportion_same_race_black
    * Variable("Proportion of people Black.ORIG")
    * Variable("BLACK")
    + c_stay_proportion_same_race_aapi
    * Variable("Proportion of people AAPI.ORIG")
    * Variable("AAPI")
    + c_stay_proportion_same_race_indian
    * Variable("Proportion of people Indian.ORIG")
    * Variable("INDIAN")
    + c_stay_proportion_same_race_latino
    * Variable("Proportion of people Latino.ORIG")
    * Variable("LATINO")
    + c_stay_proportion_foreign
    * Variable("Proportion foreign born.ORIG")
    * Variable("FOREIGN")
    # economic & housing indicators
    + c_stay_hh_income
    * Variable(
        "Median Household Income (In 2018 Inflation Adjusted Dollars).Median Household Income (In 2018 Inflation Adjusted Dollars).SE_A14006_001.ORIG"
    )
    + c_stay_proportion_struggling * Variable("Proportion of people struggling.ORIG")
    + c_stay_yrs_since_median_structure_built
    * Variable("Years since median structure built.ORIG")
    + c_stay_median_house_value_over_median_income
    * Variable("Median house value over median household income.ORIG")
    + c_stay_median_gross_rent_percentage_hh_inc
    * Variable("Median gross rent as a percentage of household income.ORIG")
    + c_stay_median_owner_costs
    * Variable(
        "Median selected monthly owner costs as percentage of household income.ORIG"
    )
    + c_stay_unemp_rate * Variable("Unemployment rate.ORIG")
    + c_stay_vacancy_rate * Variable("House vacancy proportion.ORIG")
    # work indicators
    + c_stay_median_travel_time * Variable("Median travel time.ORIG")
    + c_stay_proportion_alt_commute * Variable("Proportion alternative commute.ORIG")
    # NAICS industry group (individual-level)
    + c_stay_mil * Variable("IN_MILITARY")
    + c_stay_naics_govt * Variable("NAICS_GOVT")
    + c_stay_naics_goods_trade * Variable("NAICS_GOODS_TRADE")
    + c_stay_naics_license * Variable("NAICS_LICENSE")
    + c_stay_naics_high_ed * Variable("NAICS_HIGH_ED")
    + c_stay_naics_agr_ext * Variable("NAICS_AGR_EXT")
    # NAICS industry group (area-level proportion)
    + c_stay_proportion_mil
    * Variable("Proportion of people in military.ORIG")
    * Variable("IN_MILITARY")
    + c_stay_naics_proportion_govt
    * Variable("NAICS_GROUP_PCT_GOVT.ORIG")
    * Variable("NAICS_GOVT")
    + c_stay_naics_proportion_goods_trade
    * Variable("NAICS_GROUP_PCT_GOODS_TRADE.ORIG")
    * Variable("NAICS_GOODS_TRADE")
    + c_stay_naics_proportion_license
    * Variable("NAICS_GROUP_PCT_LICENSE.ORIG")
    * Variable("NAICS_LICENSE")
    + c_stay_naics_proportion_high_ed
    * Variable("NAICS_GROUP_PCT_HIGH_ED.ORIG")
    * Variable("NAICS_HIGH_ED")
    + c_stay_naics_proportion_agr_ext
    * Variable("NAICS_GROUP_PCT_AGR_EXT.ORIG")
    * Variable("NAICS_AGR_EXT")
)

In [ ]:
# Destination Choice Parameters to be estimated

# geography
c_destchoice_dist = Beta("c_destchoice_dist", 0, None, None, 0)
c_destchoice_logdist = Beta("c_destchoice_logdist", 0, None, None, 0)

c_destchoice_samestate = Beta("c_destchoice_samestate", 0, None, None, 0)
c_destchoice_birthstate = Beta("c_destchoice_birthstate", 0, None, None, 0)
c_destchoice_samecbsa = Beta("c_destchoice_samecbsa", 0, None, None, 0)
c_destchoice_cbsa_dist = Beta("c_destchoice_cbsa_dist", 0, None, None, 0)

# economic
c_destchoice_hh_income = Beta("c_destchoice_hh_income", 0, None, None, 0)
c_destchoice_proportion_struggling = Beta(
    "c_destchoice_proportion_struggling", 0, None, None, 0
)
c_destchoice_yrs_since_median_structure_built = Beta(
    "c_destchoice_yrs_since_median_structure_built", 0, None, None, 0
)
c_destchoice_median_house_value_over_median_income = Beta(
    "c_destchoice_median_house_value_over_median_income", 0, None, None, 0
)
c_destchoice_median_gross_rent_percentage_hh_inc = Beta(
    "c_destchoice_median_gross_rent_percentage_hh_inc", 0, None, None, 0
)
c_destchoice_median_owner_costs = Beta(
    "c_destchoice_median_owner_costs", 0, None, None, 0
)
c_destchoice_unemp_rate = Beta("c_destchoice_unemp", 0, None, None, 0)
c_destchoice_vacancy_rate = Beta("c_destchoice_vacancy_rate", 0, None, None, 0)

# similarity to migrant
c_destchoice_proportion_college = Beta(
    "c_destchoice_proportion_college", 0, None, None, 0
)
c_destchoice_proportion_foreign = Beta("c_destchoice_foreign", 0, None, None, 0)

c_destchoice_proportion_age_18_34 = Beta("c_destchoice_age_18_34", 0, None, None, 0)
c_destchoice_proportion_age_35_64 = Beta("c_destchoice_age_35_64", 0, None, None, 0)
c_destchoice_proportion_age_over_65 = Beta("c_destchoice_age_over_65", 0, None, None, 0)

c_destchoice_proportion_same_race_black = Beta(
    "c_destchoice_proportion_same_race_black", 0, None, None, 0
)
c_destchoice_proportion_same_race_aapi = Beta(
    "c_destchoice_proportion_same_race_aapi", 0, None, None, 0
)
c_destchoice_proportion_same_race_indian = Beta(
    "c_destchoice_proportion_same_race_indian", 0, None, None, 0
)
c_destchoice_proportion_same_race_latino = Beta(
    "c_destchoice_proportion_same_race_latino", 0, None, None, 0
)
c_destchoice_proportion_hh_with_children = Beta(
    "c_destchoice_proportion_hh_with_children", 0, None, None, 0
)

# quality of life
c_destchoice_proportion_ent = Beta("c_destchoice_proportion_ent", 0, None, None, 0)
c_destchoice_proportion_ent_18_34 = Beta(
    "c_destchoice_proportion_ent_18_34", 0, None, None, 0
)
c_destchoice_proportion_ent_35_64 = Beta(
    "c_destchoice_proportion_ent_35_64", 0, None, None, 0
)
# reference, wrapped into the base ent variable
# c_destchoice_proportion_ent_65 = Beta(
#     "c_destchoice_proportion_ent_65", 0, None, None, 1
# )
c_destchoice_median_travel_time = Beta(
    "c_destchoice_median_travel_time", 0, None, None, 0
)
c_dest_proportion_alt_commute = Beta(
    "c_destchoice_proportion_alt_commute", 0, None, None, 0
)

# differences between density
# c_destchoice_T34_T34 = Beta("c_destchoice_T34_T34", 0, None, None, 1)
c_destchoice_T34_metro = Beta("c_destchoice_T34_metro", 0, None, None, 0)
c_destchoice_T34_micro = Beta("c_destchoice_T34_micro", 0, None, None, 0)
c_destchoice_metro_T34 = Beta("c_destchoice_metro_T34", 0, None, None, 0)
c_destchoice_metro_metro = Beta("c_destchoice_metro_metro", 0, None, None, 0)
c_destchoice_metro_micro = Beta("c_destchoice_metro_micro", 0, None, None, 0)
c_destchoice_micro_T34 = Beta("c_destchoice_micro_T34", 0, None, None, 0)
c_destchoice_micro_metro = Beta("c_destchoice_micro_metro", 0, None, None, 0)
c_destchoice_micro_micro = Beta("c_destchoice_micro_micro", 0, None, None, 0)

# jobs
c_destchoice_proportion_military = Beta("c_destchoice_military", 0, None, None, 0)
c_destchoice_naics_proportion_govt = Beta(
    "c_destchoice_naics_proportion_govt", 0, None, None, 0
)
c_destchoice_naics_proportion_goods_trade = Beta(
    "c_destchoice_naics_proportion_goods_trade", 0, None, None, 0
)
c_destchoice_naics_proportion_license = Beta(
    "c_destchoice_proportion_naics_license", 0, None, None, 0
)
c_destchoice_naics_proportion_high_ed = Beta(
    "c_destchoice_proportion_naics_high_ed", 0, None, None, 0
)
c_destchoice_naics_proportion_agr_ext = Beta(
    "c_destchoice_proportion_naics_agr_ext", 0, None, None, 0
)

In [ ]:
# defining the utility functions for each of the moving PUMA alternatives
for i in range(1, num_alternatives + 1):
    alt = f"ALT{i}_"

    same_state = Variable("ST") == Variable(f"{alt}STATE")
    # NAME_NUM.ORIG and ALT{i}_CBSA are factorized against the same CBSA-name codebook
    # (see create_estdata.ipynb), so they're directly comparable
    same_cbsa = Variable("NAME_NUM.ORIG") == Variable(f"{alt}CBSA")
    same_type_t34 = Variable("TYPE_NUM.ORIG") == 0
    same_type_metro = Variable("TYPE_NUM.ORIG") == 1
    same_type_micro = Variable("TYPE_NUM.ORIG") == 2
    alt_type_t34 = Variable(f"{alt}TYPE") == 0
    alt_type_metro = Variable(f"{alt}TYPE") == 1
    alt_type_micro = Variable(f"{alt}TYPE") == 2

    V[i] = (
        log(Variable(f"{alt}TOT_POP"))
        # geography
        + (1 - same_cbsa)
        * (
            c_destchoice_dist * Variable(f"{alt}DIST")
            + c_destchoice_logdist * log(Variable(f"{alt}DIST") + 1)
        )
        + same_cbsa * c_destchoice_cbsa_dist * Variable(f"{alt}DIST")
        + c_destchoice_samecbsa * same_cbsa
        + c_destchoice_samestate * same_state
        + c_destchoice_birthstate * (Variable("POBP") == Variable(f"{alt}STATE"))
        # economic
        + c_destchoice_hh_income * Variable(f"{alt}HH_MED_INC")
        + c_destchoice_proportion_struggling * Variable(f"{alt}POVERTY_PCT")
        + c_destchoice_yrs_since_median_structure_built
        * Variable(f"{alt}YR_SINCE_MED_STRUCTURE")
        + c_destchoice_median_house_value_over_median_income
        * Variable(f"{alt}MED_HOUSE_VALUE_OVER_MED_HH_INC")
        + c_destchoice_median_gross_rent_percentage_hh_inc
        * Variable(f"{alt}MED_RENT_PCT_HH_INC")
        + c_destchoice_median_owner_costs * Variable(f"{alt}MED_OWNER_COST_HH_INC_PCT")
        + c_destchoice_unemp_rate * Variable(f"{alt}UNEMP_RATE")
        + c_destchoice_vacancy_rate * Variable(f"{alt}HOUSE_VACANCY_PCT")
        # similarity to migrant
        + c_destchoice_proportion_college
        * Variable("IN_COLLEGE")
        * Variable(f"{alt}COLLEGE_PCT")
        + c_destchoice_proportion_foreign
        * Variable("FOREIGN")
        * Variable(f"{alt}FOREIGN_BORN_PCT")
        + c_destchoice_proportion_age_18_34
        * Variable("AGE_18_34")
        * Variable(f"{alt}OWN_AGE_PCT")
        + c_destchoice_proportion_age_35_64
        * Variable("AGE_35_64")
        * Variable(f"{alt}OWN_AGE_PCT")
        + c_destchoice_proportion_age_over_65
        * Variable("AGE_OVER_65")
        * Variable(f"{alt}OWN_AGE_PCT")
        + c_destchoice_proportion_same_race_black
        * Variable("BLACK")
        * Variable(f"{alt}OWN_RACE_PCT")
        + c_destchoice_proportion_same_race_aapi
        * Variable("AAPI")
        * Variable(f"{alt}OWN_RACE_PCT")
        + c_destchoice_proportion_same_race_indian
        * Variable("INDIAN")
        * Variable(f"{alt}OWN_RACE_PCT")
        + c_destchoice_proportion_same_race_latino
        * Variable("LATINO")
        * Variable(f"{alt}OWN_RACE_PCT")
        + c_destchoice_proportion_hh_with_children
        * Variable(f"{alt}HH_WITH_CHILD_PCT")
        * Variable("CHILD")
        # quality of life
        + c_destchoice_proportion_ent * Variable(f"{alt}ENT_JOBS_PCT")
        + c_destchoice_proportion_ent_18_34
        * Variable("AGE_18_34")
        * Variable(f"{alt}ENT_JOBS_PCT")
        + c_destchoice_proportion_ent_35_64
        * Variable("AGE_35_64")
        * Variable(f"{alt}ENT_JOBS_PCT")
        + c_destchoice_median_travel_time * Variable(f"{alt}MED_TRAVEL_TIME")
        + c_dest_proportion_alt_commute * Variable(f"{alt}ALT_COMMUTE_PCT")
        # origin area type x destination area type
        # + c_destchoice_T34_T34 * same_type_t34 * alt_type_t34
        + c_destchoice_T34_metro * same_type_t34 * alt_type_metro
        + c_destchoice_T34_micro * same_type_t34 * alt_type_micro
        + c_destchoice_metro_T34 * same_type_metro * alt_type_t34
        + c_destchoice_metro_metro * same_type_metro * alt_type_metro
        + c_destchoice_metro_micro * same_type_metro * alt_type_micro
        + c_destchoice_micro_T34 * same_type_micro * alt_type_t34
        + c_destchoice_micro_metro * same_type_micro * alt_type_metro
        + c_destchoice_micro_micro * same_type_micro * alt_type_micro
        # jobs
        + c_destchoice_proportion_military
        * Variable("IN_MILITARY")
        * Variable(f"{alt}MIL_PCT")
        + c_destchoice_naics_proportion_govt
        * Variable("NAICS_GOVT")
        * Variable(f"{alt}NAICS_GROUP_PCT_GOVT")
        + c_destchoice_naics_proportion_goods_trade
        * Variable("NAICS_GOODS_TRADE")
        * Variable(f"{alt}NAICS_GROUP_PCT_GOODS_TRADE")
        + c_destchoice_naics_proportion_license
        * Variable("NAICS_LICENSE")
        * Variable(f"{alt}NAICS_GROUP_PCT_LICENSE")
        + c_destchoice_naics_proportion_high_ed
        * Variable("NAICS_HIGH_ED")
        * Variable(f"{alt}NAICS_GROUP_PCT_HIGH_ED")
        + c_destchoice_naics_proportion_agr_ext
        * Variable("NAICS_AGR_EXT")
        * Variable(f"{alt}NAICS_GROUP_PCT_AGR_EXT")
    )


In [ ]:
# all alternatives are available
av = {}
for i in range(num_alternatives + 1):
    av[i] = 1

In [ ]:
logger = blog.get_screen_logger(level=blog.INFO)

In [ ]:
# nest_move = move, list(range(1, num_alternatives + 1))
# nest_stay = 1.0, [0]
# nests = nest_move, nest_stay

In [ ]:
# nest_logprob = models.lognested(V, av, nests, CHOSEN)

# biogeme_nest = bio.BIOGEME(database, nest_logprob, suggestScales=False)
# biogeme_nest.modelName = "nested_full_full"

# biogeme_nest.calculateNullLoglikelihood(av)

# results_nest = biogeme_nest.estimate()
# pandasResults_nest = results_nest.getEstimatedParameters()
# print(pandasResults_nest)

In [ ]:
# Definition of the model. This is the contribution of each
# observation to the log likelihood function.
# estimating the CHOSEN field
logprob = models.loglogit(V, av, Variable("ALT_CHOICE"))

# formulas = {"loglike": logprob, "weight": W0}

path = os.path.join("results", f"us_mnl_{year}_{unixtime}")
os.makedirs(path, exist_ok=True)

# Create the Biogeme object
biogeme = bio.BIOGEME(database, logprob)
biogeme.model_name = os.path.join(path, "model")

In [ ]:
# Calculate the null log likelihood for reporting. (likelihood of predicting every entry's alterantive correctly if alternatives are randomly chosen)
biogeme.calculate_null_loglikelihood(av)

In [ ]:
# estimate parameters
results = biogeme.estimate()

# Get the results in a pandas table
pandasResults = results.get_estimated_parameters()
print(pandasResults)